In [11]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END 
from langchain.chat_models import init_chat_model 
from pydantic import BaseModel 

llm = init_chat_model("openai:gpt-4o")

In [12]:
class State(TypedDict):
    dish : str
    ingredients : list[dict]
    recipe_step : str 
    plating_instructions : str
    
class Ingredient(BaseModel): # 재료 
    name : str
    quantity :  str # 양 
    unit : str #단위 

class IngredientsOutput(BaseModel):
    ingredients : List[Ingredient]

In [13]:
def list_ingredients(state: State):
    # 이렇게 하면 llm을 호출할 떄 이 출력형식으로로 강제화 가능하다, 출력은 재료들의 리스트이고 재료들은  Ingredient 이 함수 형식을 따라야함 
    structured_llm = llm.with_structured_output(IngredientsOutput) 
    response = structured_llm.invoke(
        f"List 5-8 ingredients needed to make {state['dish']}"
    )
    return {"ingredients" : response.ingredients} # 재료들을 state 에 넣는다.

def create_recipe(state: State):
    # 이 재료들을 사용해서 state를 위한 단계별 요리법을 작성해 
    response = llm.invoke(f"Write a step by step cooking instruction for {state["dish"]}, using these ingredients {state['ingredients']}")
    return {
        "recipe_step" : response.content # 구조화된 출력이 아닐떄는 llm에서 일반 메시지가 나오니깐 메시지 안에content필드로 들어가도록 
    }

def describe_plating(state: State):
    # 이 레시피를 기반으로 해서 이 요리를어떻게 아름답게 플레이팅할지 설명해 
    response = llm.invoke(f"Describe how to beautifully plate this dish. {state["dish"]} based on this recipe {state["recipe_step"]}")
    return {
        "plating_instructions" : response.content
    }

In [14]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe",create_recipe )
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()


In [15]:
graph.invoke({"dish" : "hummus"})

c:\Users\gkrud\Documents\workflow-architectures\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=IngredientsOutput(ingredi...ust for consistency)')]), input_type=IngredientsOutput])
  return self.__pydantic_serializer__.to_python(


{'dish': 'hummus',
 'ingredients': [Ingredient(name='Chickpeas', quantity='1', unit='cup (canned or cooked)'),
  Ingredient(name='Tahini', quantity='1/4', unit='cup'),
  Ingredient(name='Lemon juice', quantity='2-3', unit='tablespoons'),
  Ingredient(name='Garlic', quantity='1-2', unit='cloves (minced)'),
  Ingredient(name='Olive oil', quantity='1-2', unit='tablespoons'),
  Ingredient(name='Salt', quantity='1/2', unit='teaspoon'),
  Ingredient(name='Ground cumin', quantity='1', unit='teaspoon'),
  Ingredient(name='Water', quantity='2-3', unit='tablespoons (adjust for consistency)')],
 'recipe_step': "Here's a step-by-step cooking instruction for hummus using the given ingredients:\n\n**Ingredients:**\n- 1 cup Chickpeas (canned or cooked)\n- 1/4 cup Tahini\n- 2-3 tablespoons Lemon juice\n- 1-2 cloves Garlic (minced)\n- 1-2 tablespoons Olive oil\n- 1/2 teaspoon Salt\n- 1 teaspoon Ground cumin\n- 2-3 tablespoons Water (adjust for consistency)\n\n**Instructions:**\n\n1. **Prepare Chickpeas